In [ ]:
# ==================================================
# SmogNet
# 07_Gold_Layer_Creation
# Final Business Layer
# ==================================================

import pandas as pd
import numpy as np

anomaly=spark.sql("""

SELECT * FROM Anomaly_output

""").toPandas()

classification=spark.sql("""

SELECT * FROM Classification_output

""").toPandas()

alerts=spark.sql("""

SELECT * FROM Alert_output

""").toPandas()

print(anomaly.shape)

print(classification.shape)

print(alerts.shape)

StatementMeta(, 9e96171d-15be-4ab3-b3df-a06bd537c11d, 4, Finished, Available, Finished, False)

(21792, 59)
(1090, 61)
(1090, 62)


In [3]:
Gold_anomalies=anomaly[

[
"parsed_datetime",
"city",
"components_pm2_5",
"components_pm10",
"anomaly_score",
"risk_level",
"severity_score"

]

].copy()

Gold_anomalies.head()

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 6, Finished, Available, Finished, False)

,parsed_datetime,city,components_pm2_5,components_pm10,anomaly_score,risk_level,severity_score
0,2024-08-17 00:00:00,Islamabad,12.12,12.86,0.038258,Low,9.306
1,2024-08-17 01:00:00,Islamabad,11.15,11.83,0.060344,Low,8.609
2,2024-08-17 02:00:00,Islamabad,11.17,12.12,0.080123,Low,8.704
3,2024-08-17 03:00:00,Islamabad,12.23,13.52,0.046535,Low,9.548
4,2024-08-17 04:00:00,Islamabad,12.31,13.75,0.029080,Low,9.649


In [4]:
spark.createDataFrame(
Gold_anomalies
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Gold_anomalies"
)

print(
"Gold_anomalies saved"
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 8, Finished, Available, Finished, False)

Gold_anomalies saved


In [9]:
Gold_classification=classification[

[
"parsed_datetime",
"city",
"predicted_source",
"confidence"
]

].copy()

Gold_classification.head()

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 25, Finished, Available, Finished, False)

,parsed_datetime,city,predicted_source,confidence
0,2024-06-09 16:00:00,Islamabad,Crop Burning,0.65
1,2024-06-09 17:00:00,Islamabad,Crop Burning,0.65
2,2024-06-09 18:00:00,Islamabad,Crop Burning,0.65
3,2024-06-09 19:00:00,Islamabad,Crop Burning,0.65
4,2024-06-09 20:00:00,Islamabad,Crop Burning,0.65


In [6]:
spark.createDataFrame(
Gold_classification
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Gold_classification"
)

print(
"Gold_classification saved"
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 11, Finished, Available, Finished, False)

Gold_classification saved


In [10]:
Gold_alerts=alerts[

[
"parsed_datetime",
"city",
"predicted_source",
"severity",
"alert_message"
]

].copy()

Gold_alerts.head()

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 28, Finished, Available, Finished, False)

,parsed_datetime,city,predicted_source,severity,alert_message
0,2024-12-31 13:00:00,Peshawar,Crop Burning,Low,Air quality in Peshawar has shown unusual chan...
1,2024-12-31 14:00:00,Peshawar,Crop Burning,Low,Air quality in Peshawar has shown unusual chan...
2,2024-12-31 15:00:00,Peshawar,Crop Burning,Low,Air quality in Peshawar has shown unusual chan...
3,2024-12-31 16:00:00,Peshawar,Crop Burning,Low,Air quality in Peshawar has shown unusual chan...
4,2024-12-31 17:00:00,Peshawar,Crop Burning,Low,Air quality in Peshawar has shown unusual chan...


In [11]:
spark.createDataFrame(
Gold_alerts
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Gold_alerts"
)

print(
"Gold_alerts saved"
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 30, Finished, Available, Finished, False)

Gold_alerts saved


In [12]:
summary=pd.DataFrame({

"Total_Anomalies":[anomaly["anomaly_flag"].sum() 
],

"Critical_Alerts":[

len(

alerts[
alerts[
"severity"
]=="Critical"
]

)

],

"Cities":[

alerts[
"city"
].nunique()

],

"Average_Severity":[

alerts[
"anomaly_score"
].mean()

]

})

summary

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 31, Finished, Available, Finished, False)

,Total_Anomalies,Critical_Alerts,Cities,Average_Severity
0,1090,0,5,0.24663


In [13]:
spark.createDataFrame(
summary
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Gold_dashboard_summary"
)

print(
"Summary saved"
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 32, Finished, Available, Finished, False)

Summary saved


In [14]:
city_risk=alerts.groupby(

"city"

).agg({

"anomaly_score":"mean"

}).reset_index()

city_risk.columns=[

"city",

"risk_index"

]

city_risk=city_risk.sort_values(

"risk_index",

ascending=False

)

city_risk[
"rank"
]=range(

1,

len(
city_risk
)+1

)

city_risk

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 33, Finished, Available, Finished, False)

,city,risk_index,rank
1,Karachi,0.309878,1
2,Lahore,0.256720,2
4,Quetta,0.253868,3
3,Peshawar,0.221551,4
0,Islamabad,0.214023,5


In [15]:
spark.createDataFrame(
city_risk
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Gold_city_risk"
)

print(
"Gold_city_risk saved"
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 34, Finished, Available, Finished, False)

Gold_city_risk saved


In [16]:
spark.sql(
"SHOW TABLES"
).show(
100,
False
)

StatementMeta(, 542e3772-5753-4d4d-a034-248e10c86bbd, 35, Finished, Available, Finished, False)

+---------------------------------------+----------------------+-----------+
|namespace                              |tableName             |isTemporary|
+---------------------------------------+----------------------+-----------+
|SmogNet_Datathon.Datathon_Lakehouse.dbo|alert_output          |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|anomaly_output        |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|bronze_testing        |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|bronze_training       |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|classification_output |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|eda_dataset           |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|gold_alerts           |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|gold_anomalies        |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|gold_city_risk        |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|gold_classification   |false      |